# 🌐 Análisis de la Brecha Digital y Calidad de Red en Chile
## Etapa II: Avance del Proyecto – Modelo Preliminar

**Proyecto Semestral – Data Science**  
**Ramo:** Data Science | **Profesor:** Jorge Rozas  
**Integrantes:** Diego Lara, Cristóbal Castillo  
**Fecha:** Octubre 2026

---

## Objetivo del Notebook

Este notebook documenta el **avance de la Etapa II** del proyecto semestral de Data Science, cubriendo las siguientes etapas del pipeline CRISP-DM:

1. **Comprensión del problema** y contexto de la brecha digital en Chile
2. **Integración y preparación** de datos comunales desde fuentes oficiales (SUBTEL, INE, CASEN)
3. **Análisis Exploratorio de Datos (EDA)** con hallazgos clave
4. **Modelo de Clustering** (K-Means) para segmentar comunas por vulnerabilidad digital
5. **Modelo de Clasificación** (Regresión Logística + Random Forest) como baseline predictivo

Todo el código es **completamente reproducible** con semilla fija `RANDOM_SEED=42`.

---
# SECCIÓN 1: Marco Metodológico – CRISP-DM

El proyecto adopta la metodología **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*), el estándar más utilizado en la industria para proyectos de Data Science. Esta metodología es iterativa y permite que los hallazgos de una fase informen las decisiones de fases anteriores.

```
┌───────────────────────────────────────────────────────────┐
│                         CRISP-DM                          │
│                                                           │
│  1. Comprensión del Negocio  ──► ¿Por qué la brecha?     │
│  2. Comprensión de los Datos ──► SUBTEL + INE + CASEN     │
│  3. Preparación de Datos     ──► Limpieza, merge, KPIs    │
│  4. Modelado                 ──► Clustering + Clasif.     │
│  5. Evaluación               ──► Métricas, validación     │
│  6. Despliegue               ──► (Etapa III/IV)           │
└───────────────────────────────────────────────────────────┘
```

### Justificación del enfoque híbrido (No Supervisado + Supervisado)

| Fase | Algoritmo | Objetivo |
|------|-----------|----------|
| **1. Clustering** | K-Means | Descubrir perfiles naturales de vulnerabilidad **sin etiquetas previas** |
| **2. Clasificación** | RF + LR | Predecir a qué perfil pertenece una nueva comuna y **encontrar los factores clave** |

---
# SECCIÓN 2: Configuración del Entorno

In [ ]:
# ============================================================
# LIBRERÍAS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import sys
import joblib

warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, silhouette_score
)

# ============================================================
# REPRODUCIBILIDAD
# ============================================================
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ============================================================
# RUTAS (relativas al notebook)
# ============================================================
BASE_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PROC = os.path.join(BASE_DIR, 'data', 'processed')
DATA_RAW  = os.path.join(BASE_DIR, 'data', 'raw')
FIGURES   = os.path.join(BASE_DIR, 'outputs', 'figures')
MODELS    = os.path.join(BASE_DIR, 'outputs', 'models')

for path in [FIGURES, MODELS]:
    os.makedirs(path, exist_ok=True)

# ============================================================
# ESTÉTICA
# ============================================================
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.3})
sns.set_style('whitegrid')
sns.set_palette('viridis')

print(f'Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}')
print(f'Base dir: {BASE_DIR}')
print('Entorno configurado correctamente')

---
# SECCIÓN 3: Fuentes de Datos y Carga

## Fuentes oficiales utilizadas

| Fuente | Organismo | URL | Variables clave |
|--------|-----------|-----|-----------------|
| **SUBTEL** | Subsecretaría de Telecomunicaciones | subtel.gob.cl | Conexiones fijas residenciales por comuna, 2015–2026 |
| **INE** | Instituto Nacional de Estadísticas | ine.gob.cl | Proyecciones de población 2024 por CUT comunal |
| **CASEN** | Min. Desarrollo Social | observatorio.ministeriodesarrollosocial.gob.cl | Tasa de pobreza por ingresos y multidimensional 2022 por comuna |

### Proceso de integración

La **llave de unión** es el **Código Único Territorial (CUT)**, un identificador de 4–5 dígitos que el INE asigna a cada una de las 346 comunas de Chile. SUBTEL y CASEN también usan este código como referencia.

```
SUBTEL (346 comunas oficiales) ──┐
                          ├── Merge por nombre_comuna / CUT ──► Dataset Final (346 comunas x 13 cols)
INE   (346 comunas)   ──┤
                          │
CASEN (345 comunas)   ──┘
```

In [ ]:
# ============================================================
# CARGA DEL DATASET INTEGRADO
# El dataset fue generado por src/merge_datos.py integrando:
#   - SUBTEL: series_internet_fija_subtel.xlsx
#   - INE:    ine_proyecciones_poblacion_comunas.xlsx
#   - CASEN:  casen_2022_pobreza_ingresos_comunas.xlsx
#             casen_2022_pobreza_multidimensional_comunas.xlsx
# ============================================================

csv_path = os.path.join(DATA_PROC, 'dataset_comunas_chile.csv')
df = pd.read_csv(csv_path, encoding='utf-8-sig').drop_duplicates(subset=['cut_comuna'], keep='first').reset_index(drop=True)

print(f'Dataset cargado: {df.shape[0]} comunas x {df.shape[1]} columnas')
print(f'\nColumnas disponibles:')
for col in df.columns:
    print(f'  - {col}: {df[col].dtype} | Nulos: {df[col].isnull().sum()} ({df[col].isnull().mean()*100:.1f}%)')

print(f'\nPrimeras 5 filas:')
df.head(5)

In [ ]:
# ============================================================
# DESCRIPCIÓN DE VARIABLES
# ============================================================

descripcion_vars = {
    'num_region':                    'Número de región (1 a 16)',
    'cut_comuna':                    'Código Único Territorial INE (clave de unión)',
    'nombre_comuna':                 'Nombre oficial de la comuna',
    'poblacion_2024':                'Proyección de población total 2024 (INE)',
    'poblacion_2023':                'Proyección de población total 2023 (INE)',
    'conexiones_recientes':          'Conexiones fijas residenciales – último mes disponible (SUBTEL Jun 2026)',
    'conexiones_prom2024':           'Promedio mensual de conexiones fijas residenciales – año 2024 (SUBTEL)',
    'conexiones_prom2023':           'Promedio mensual de conexiones fijas residenciales – año 2023 (SUBTEL)',
    'tasa_pobreza_ingresos':         'Tasa de pobreza por ingresos 2022 – estimación SAE (CASEN)',
    'tasa_pobreza_multidimensional': 'Tasa de pobreza multidimensional 2022 – estimación SAE (CASEN)',
    'conexiones_x1000hab':           'Indicador KPI: conexiones fijas por cada 1.000 habitantes (SUBTEL/INE)',
    'es_rural':                      'Proxy de ruralidad: 1=Rural (pob<15.000), 0=Urbana',
    'pct_ruralidad_proxy':           'Proxy: % ruralidad estimado por tamaño de población',
}

pd.DataFrame(descripcion_vars.items(), columns=['Variable', 'Descripción'])

---
# SECCIÓN 4: Preparación de Datos

## Decisiones metodológicas de limpieza

| Problema | Estrategia aplicada | Justificación |
|----------|--------------------|--------------|
| **Missing values** | Imputación con mediana | Robusta a outliers; preserva la distribución |
| **Outliers** | Winsorización p1–p99 | Evita eliminar comunas legítimas con valores extremos |
| **Escalas distintas** | StandardScaler (clustering), MinMaxScaler (clasificación) | El clustering requiere media=0, std=1; la clasificación se beneficia de [0,1] |

In [ ]:
# ============================================================
# ANÁLISIS DE VALORES FALTANTES
# ============================================================
cols_num = df.select_dtypes(include=[np.number]).columns.tolist()
nulos = df[cols_num].isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(1)

df_nulos = pd.DataFrame({'N Nulos': nulos, '% Nulos': pct_nulos}).sort_values('% Nulos', ascending=False)
df_nulos = df_nulos[df_nulos['N Nulos'] > 0]

print('Variables con valores faltantes:')
print(df_nulos.to_string())

# Visualización de nulos
if len(df_nulos) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#e74c3c' if p > 20 else '#f39c12' if p > 10 else '#27ae60' for p in df_nulos['% Nulos']]
    ax.barh(df_nulos.index, df_nulos['% Nulos'], color=colors, edgecolor='white')
    ax.set_xlabel('% de valores faltantes')
    ax.set_title('Porcentaje de Valores Faltantes por Variable', fontweight='bold')
    ax.axvline(5, color='orange', linestyle='--', alpha=0.7, label='5%')
    ax.axvline(20, color='red', linestyle='--', alpha=0.7, label='20%')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'\nTotal comunas con datos SUBTEL: {df["conexiones_recientes"].notna().sum()}/346')
    print(f'Total comunas con datos CASEN:  {df["tasa_pobreza_ingresos"].notna().sum()}/346')

In [ ]:
# ============================================================
# DETECCIÓN Y TRATAMIENTO DE OUTLIERS (IQR)
# ============================================================
def reporte_outliers_iqr(df, cols):
    """Detecta outliers con el método IQR y reporta su magnitud."""
    rows = []
    for col in cols:
        d = df[col].dropna()
        Q1, Q3 = d.quantile(0.25), d.quantile(0.75)
        IQR = Q3 - Q1
        n_out = ((d < Q1 - 1.5*IQR) | (d > Q3 + 1.5*IQR)).sum()
        rows.append({'Variable': col, 'N Outliers': n_out,
                     '% Outliers': round(n_out/len(d)*100, 1),
                     'Límite Inf': round(Q1 - 1.5*IQR, 2),
                     'Límite Sup': round(Q3 + 1.5*IQR, 2)})
    return pd.DataFrame(rows).sort_values('% Outliers', ascending=False)

cols_analizar = ['conexiones_x1000hab', 'conexiones_prom2024',
                 'poblacion_2024', 'tasa_pobreza_ingresos']
cols_analizar = [c for c in cols_analizar if c in df.columns]

df_outliers = reporte_outliers_iqr(df, cols_analizar)
print('Reporte de Outliers (Método IQR):')
display(df_outliers)

print('\nEstrategia: Winsorización p1–p99 para preservar todas las comunas')
print('(Las comunas con valores extremos son reales: Santiago tiene 700k hab,') 
print(' mientras Colchane tiene 1.5k; ambas son legítimas)')

In [ ]:
# ============================================================
# ESTADÍSTICAS DESCRIPTIVAS
# ============================================================
print('Estadísticas descriptivas de variables clave:')
display(df[cols_analizar].describe().round(3))

# Hallazgos rápidos
d_con = df['conexiones_x1000hab'].dropna()
d_pob = df['tasa_pobreza_ingresos'].dropna()

print(f'\nHALLAZGOS CLAVE:')
print(f'  - Mediana conexiones/1000 hab: {d_con.median():.1f}  |  Media: {d_con.mean():.1f}')
print(f'  - Rango: [{d_con.min():.1f} – {d_con.max():.0f}] → variación de {d_con.max()/d_con.min():.0f}x entre comunas')
print(f'  - Mediana tasa pobreza ingresos: {d_pob.median()*100:.1f}%')
print(f'  - Max tasa pobreza: {d_pob.max()*100:.1f}% | Min: {d_pob.min()*100:.1f}%')

# Comunas con mayor/menor conectividad
top5 = df.nlargest(5, 'conexiones_x1000hab')[['nombre_comuna', 'num_region', 'conexiones_x1000hab', 'tasa_pobreza_ingresos']]
bot5 = df.nsmallest(5, 'conexiones_x1000hab')[['nombre_comuna', 'num_region', 'conexiones_x1000hab', 'tasa_pobreza_ingresos']]

print(f'\nTop 5 comunas – MAYOR conectividad:')
display(top5)
print(f'Top 5 comunas – MENOR conectividad:')
display(bot5)

---
# SECCIÓN 5: Análisis Exploratorio de Datos (EDA)

## Hipótesis a validar

| Hipótesis | Variable X | Variable Y | Resultado esperado |
|-----------|-----------|-----------|--------------------|
| H1 | % Ruralidad | Conexiones/1000 hab | Correlación **negativa** |
| H2 | Tasa de pobreza | Conexiones/1000 hab | Correlación **negativa** |
| H3 | Población | Conexiones/1000 hab | Correlación **positiva** |
| H4 | Pobreza ingresos | Pobreza multidimensional | Correlación **positiva** alta |

In [ ]:
# ============================================================
# EDA 1: DISTRIBUCIONES DE VARIABLES CLAVE
# ============================================================
vars_plot = [c for c in ['conexiones_x1000hab', 'poblacion_2024',
                          'tasa_pobreza_ingresos', 'tasa_pobreza_multidimensional',
                          'conexiones_prom2024'] if c in df.columns]

n = len(vars_plot)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4*nrows))
axes = np.array(axes).flatten()
colors = plt.cm.viridis(np.linspace(0.2, 0.85, n))

for i, (var, color) in enumerate(zip(vars_plot, colors)):
    data = df[var].dropna()
    axes[i].hist(data, bins=30, color=color, alpha=0.75, edgecolor='white', linewidth=0.5)
    axes[i].axvline(data.median(), color='red', linestyle='--', lw=2,
                    label=f'Mediana: {data.median():.1f}')
    axes[i].axvline(data.mean(), color='orange', linestyle='--', lw=2,
                    label=f'Media: {data.mean():.1f}')
    axes[i].set_title(var, fontweight='bold', fontsize=10)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')
    axes[i].legend(fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de Variables Clave – 346 Comunas de Chile', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, '01_distribuciones.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Observación: "conexiones_x1000hab" tiene distribución muy sesgada a la derecha → alta desigualdad')

In [ ]:
# ============================================================
# EDA 2: HEATMAP DE CORRELACIONES
# ============================================================
corr_vars = [c for c in ['conexiones_x1000hab', 'conexiones_prom2024',
                          'poblacion_2024', 'tasa_pobreza_ingresos',
                          'tasa_pobreza_multidimensional', 'pct_ruralidad_proxy'] if c in df.columns]

corr_matrix = df[corr_vars].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'label': 'Pearson r'}, ax=ax)
ax.set_title('Matriz de Correlación de Pearson – Variables de Brecha Digital', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, '02_heatmap_correlacion.png'), dpi=150, bbox_inches='tight')
plt.show()

# Correlaciones más fuertes
corr_stack = corr_matrix.abs().unstack().sort_values(ascending=False)
corr_stack = corr_stack[corr_stack < 1.0]
print('Top 5 correlaciones más fuertes:')
print(corr_stack.head(5).to_string())
print('\nInterpretación de hipótesis:')
if 'tasa_pobreza_ingresos' in corr_vars and 'conexiones_x1000hab' in corr_vars:
    r_h2 = corr_matrix.loc['tasa_pobreza_ingresos','conexiones_x1000hab']
    print(f'  H2 (pobreza vs conectividad): r={r_h2:.3f} → {"VALIDADA" if r_h2 < -0.1 else "No validada"}')

In [ ]:
# ============================================================
# EDA 3: SCATTER – POBREZA vs. CONECTIVIDAD
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Pobreza por ingresos vs conexiones/1000 hab
df_plot = df.dropna(subset=['tasa_pobreza_ingresos', 'conexiones_x1000hab'])

sc = axes[0].scatter(
    df_plot['tasa_pobreza_ingresos'] * 100,
    np.log1p(df_plot['conexiones_x1000hab']),  # Log para mejor visualización
    c=df_plot['num_region'], cmap='viridis',
    alpha=0.6, s=40, edgecolors='white', linewidth=0.3
)
plt.colorbar(sc, ax=axes[0], label='Región')
axes[0].set_xlabel('Tasa de Pobreza por Ingresos 2022 (%)', fontsize=11)
axes[0].set_ylabel('log(Conexiones / 1000 hab)', fontsize=11)
axes[0].set_title('Pobreza vs. Conectividad\n(cada punto = 1 comuna)', fontsize=11, fontweight='bold')

# Anotar extremos
top3 = df_plot.nlargest(3, 'conexiones_x1000hab')
bot3 = df_plot.nsmallest(3, 'conexiones_x1000hab')
for _, row in pd.concat([top3, bot3]).iterrows():
    axes[0].annotate(row['nombre_comuna'],
                     (row['tasa_pobreza_ingresos']*100, np.log1p(row['conexiones_x1000hab'])),
                     fontsize=7, alpha=0.85)

# Plot 2: Ranking top/bottom comunas
df_sort = df.dropna(subset=['conexiones_x1000hab']).sort_values('conexiones_x1000hab')
top10 = df_sort.tail(10)
bot10 = df_sort.head(10)
combined = pd.concat([bot10.assign(tipo='Menor Conectividad'),
                       top10.assign(tipo='Mayor Conectividad')])
colors_bar = ['#e74c3c' if t == 'Menor Conectividad' else '#27ae60' for t in combined['tipo']]
axes[1].barh(combined['nombre_comuna'], combined['conexiones_x1000hab'],
             color=colors_bar, edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Conexiones por 1000 hab. (2024)', fontsize=11)
axes[1].set_title('Ranking de Conectividad Comunal\nRojo=Menor | Verde=Mayor', fontsize=11, fontweight='bold')

plt.suptitle('Análisis de Brecha Digital por Comuna – SUBTEL + CASEN 2022', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, '03_eda_pobreza_conectividad.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# EDA 4: ANÁLISIS POR REGIÓN
# ============================================================
df_region = df.groupby('num_region').agg(
    n_comunas=('nombre_comuna', 'count'),
    conectividad_mediana=('conexiones_x1000hab', 'median'),
    pobreza_media=('tasa_pobreza_ingresos', 'mean'),
    poblacion_total=('poblacion_2024', 'sum')
).round(3).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

orden_con = df_region.sort_values('conectividad_mediana')['num_region'].astype(str)
axes[0].barh(orden_con, df_region.set_index('num_region').loc[
    df_region.sort_values('conectividad_mediana')['num_region'], 'conectividad_mediana'].values,
    color=plt.cm.viridis(np.linspace(0.2, 0.9, len(df_region))), edgecolor='white')
axes[0].set_xlabel('Mediana conexiones/1000 hab.', fontsize=11)
axes[0].set_ylabel('Región', fontsize=11)
axes[0].set_title('Conectividad Mediana por Región', fontweight='bold')

orden_pob = df_region.sort_values('pobreza_media')['num_region'].astype(str)
axes[1].barh(orden_pob, df_region.set_index('num_region').loc[
    df_region.sort_values('pobreza_media')['num_region'], 'pobreza_media'].values * 100,
    color=plt.cm.Reds(np.linspace(0.3, 0.85, len(df_region))), edgecolor='white')
axes[1].set_xlabel('Tasa de Pobreza Promedio (%)', fontsize=11)
axes[1].set_ylabel('Región', fontsize=11)
axes[1].set_title('Pobreza por Ingresos Promedio por Región', fontweight='bold')

plt.suptitle('Análisis Regional – Brecha Digital en Chile', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, '04_analisis_regional.png'), dpi=150, bbox_inches='tight')
plt.show()

print('Resumen por región:')
display(df_region.sort_values('conectividad_mediana', ascending=False))

In [ ]:
# ============================================================
# EDA 5: ANÁLISIS URBANO vs. RURAL (proxy)
# ============================================================
df_ur = df.copy()
df_ur['tipo_comuna'] = df_ur['es_rural'].map({1: 'Rural (<15k hab)', 0: 'Urbana/Semiurbana'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter ruralidad vs conectividad
for tipo, color in [('Rural (<15k hab)', '#e74c3c'), ('Urbana/Semiurbana', '#3498db')]:
    mask = df_ur['tipo_comuna'] == tipo
    data = df_ur[mask].dropna(subset=['pct_ruralidad_proxy', 'conexiones_x1000hab'])
    axes[0].scatter(data['pct_ruralidad_proxy'], np.log1p(data['conexiones_x1000hab']),
                    alpha=0.5, s=25, color=color, label=f'{tipo} (n={mask.sum()})')
axes[0].set_xlabel('% Ruralidad estimado', fontsize=11)
axes[0].set_ylabel('log(Conexiones/1000 hab)', fontsize=11)
axes[0].set_title('Ruralidad vs. Conectividad', fontweight='bold')
axes[0].legend()

# Boxplot
df_box = df_ur.dropna(subset=['conexiones_x1000hab'])
sns.boxplot(data=df_box, x='tipo_comuna', y='conexiones_x1000hab',
            palette={'Rural (<15k hab)': '#e74c3c', 'Urbana/Semiurbana': '#3498db'}, ax=axes[1])
axes[1].set_title('Distribución de Conectividad\nUrbano vs. Rural', fontweight='bold')
axes[1].set_xlabel('Tipo de Comuna')
axes[1].set_ylabel('Conexiones por 1000 hab.')

# Test estadístico
from scipy import stats
rural = df_box[df_box['es_rural']==1]['conexiones_x1000hab'].dropna()
urbana = df_box[df_box['es_rural']==0]['conexiones_x1000hab'].dropna()
t_stat, p_val = stats.mannwhitneyu(rural, urbana, alternative='less')
axes[1].set_title(f'Distribución de Conectividad\nUrbano vs. Rural\n(Mann-Whitney p={p_val:.4f})', fontweight='bold')

plt.suptitle('Brecha Digital: Comunas Urbanas vs. Rurales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, '05_urbano_vs_rural.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Mediana conectividad Urbana: {urbana.median():.1f} | Rural: {rural.median():.1f}')
print(f'Mann-Whitney p-value: {p_val:.6f}')
print(f'  → {"DIFERENCIA ESTADÍSTICAMENTE SIGNIFICATIVA" if p_val < 0.05 else "Sin significancia estadística"}')

---
# SECCIÓN 6: Modelo de Clustering – K-Means

## Justificación

Usamos **K-Means** para descubrir la estructura natural de vulnerabilidad digital sin imponer etiquetas previas. K-Means minimiza la varianza intra-cluster:

$$\text{argmin}_{C} \sum_{k=1}^{K} \sum_{x \in C_k} \|x - \mu_k\|^2$$

## Selección de k
Usamos dos criterios complementarios:
- **Método del Codo:** Busca el punto de inflexión en la curva de inercia
- **Silhouette Score:** Mide cohesión intra-cluster y separación inter-cluster (rango [-1, 1], mayor = mejor)

In [ ]:
# ============================================================
# PREPARAR FEATURES PARA CLUSTERING
# ============================================================
FEATURE_COLS = [c for c in [
    'conexiones_x1000hab',
    'conexiones_prom2024',
    'tasa_pobreza_ingresos',
    'poblacion_2024',
    'pct_ruralidad_proxy',
] if c in df.columns]

print(f'Features para clustering: {FEATURE_COLS}')

df_model = df[FEATURE_COLS + ['cut_comuna', 'nombre_comuna', 'num_region']].copy()

# Imputación por mediana
for col in FEATURE_COLS:
    med = df_model[col].median()
    n_imp = df_model[col].isnull().sum()
    if n_imp > 0:
        df_model[col] = df_model[col].fillna(med)
        print(f'  Imputados {n_imp} nulos en "{col}" con mediana={med:.2f}')

# Winsorización p1-p99
for col in FEATURE_COLS:
    p1, p99 = df_model[col].quantile([0.01, 0.99])
    df_model[col] = df_model[col].clip(lower=p1, upper=p99)

# Normalización
scaler_cl = StandardScaler()
X = df_model[FEATURE_COLS].values
X_scaled = scaler_cl.fit_transform(X)

print(f'\nDataset para clustering: {X_scaled.shape} (comunas x features)')
print(f'Estadísticas post-normalización: media≈{X_scaled.mean():.2f}, std≈{X_scaled.std():.2f}')

In [ ]:
# ============================================================
# MÉTODO DEL CODO + SILHOUETTE
# ============================================================
ks = range(2, 9)
inercias, silhouettes = [], []

for k in ks:
    km = KMeans(n_clusters=k, init='k-means++', n_init=15, max_iter=300, random_state=RANDOM_SEED)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    silhouettes.append(sil)
    print(f'k={k}: Inercia={km.inertia_:,.1f} | Silhouette={sil:.4f}')

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(ks), inercias, 'bo-', lw=2.5, ms=9)
axes[0].fill_between(list(ks), inercias, alpha=0.1, color='blue')
axes[0].set_xlabel('Número de Clusters (k)', fontsize=11)
axes[0].set_ylabel('Inercia (WCSS)', fontsize=11)
axes[0].set_title('Método del Codo', fontweight='bold', fontsize=12)
axes[0].set_xticks(list(ks))

best_k = list(ks)[int(np.argmax(silhouettes))]
axes[1].plot(list(ks), silhouettes, 'rs-', lw=2.5, ms=9)
axes[1].fill_between(list(ks), silhouettes, alpha=0.1, color='red')
axes[1].axvline(best_k, color='green', linestyle='--', lw=2.5,
                label=f'Mejor k={best_k} (score={max(silhouettes):.3f})')
axes[1].set_xlabel('Número de Clusters (k)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_title('Silhouette Score (mayor = mejor separación)', fontweight='bold', fontsize=12)
axes[1].set_xticks(list(ks))
axes[1].legend(fontsize=11)

plt.suptitle('Selección del Número Óptimo de Clusters – Brecha Digital Chile', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, '06_elbow_silhouette.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\nDecisión: k={best_k} (mayor Silhouette Score: {max(silhouettes):.4f})')
N_CLUSTERS = 3  # Fijado en 3 para cumplir el Objetivo Específico 3 (Alta, Media, Baja Vulnerabilidad)

In [ ]:
# ============================================================
# K-MEANS FINAL
# ============================================================
km_final = KMeans(n_clusters=N_CLUSTERS, init='k-means++', n_init=20,
                  max_iter=500, random_state=RANDOM_SEED)
labels = km_final.fit_predict(X_scaled)
df_model['cluster'] = labels

sil_final = silhouette_score(X_scaled, labels)
print(f'Silhouette Score del modelo final (k={N_CLUSTERS}): {sil_final:.4f}')
print(f'\nDistribución de comunas por cluster:')
print(pd.Series(labels).value_counts().sort_index().to_string())

# Perfiles de clusters
centroides = df_model.groupby('cluster')[FEATURE_COLS].mean().round(3)
print(f'\nCentroides por cluster:')
display(centroides)

# Asignar etiquetas semánticas
if 'conexiones_x1000hab' in centroides.columns:
    rank = centroides['conexiones_x1000hab'].rank()
    etiquetas_cluster = {}
    for c, r in rank.items():
        if r == rank.max():    etiquetas_cluster[c] = 'Baja Vulnerabilidad'
        elif r == rank.min(): etiquetas_cluster[c] = 'Alta Vulnerabilidad'
        else:                  etiquetas_cluster[c] = 'Vulnerabilidad Media'
    df_model['perfil'] = df_model['cluster'].map(etiquetas_cluster)
    print(f'\nEtiquetas asignadas: {etiquetas_cluster}')

# Guardar modelo
joblib.dump(km_final, os.path.join(MODELS, 'kmeans_model.pkl'))
print('Modelo guardado en outputs/models/kmeans_model.pkl')

In [ ]:
# ============================================================
# VISUALIZACIÓN DE CLUSTERS: PCA 2D + HEATMAP DE PERFILES
# ============================================================
pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_
print(f'PCA: PC1={var_exp[0]:.1%} | PC2={var_exp[1]:.1%} | Total={sum(var_exp):.1%}')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
COLORES = ['#e74c3c', '#f39c12', '#27ae60', '#3498db', '#9b59b6']

for k in sorted(df_model['cluster'].unique()):
    mask = df_model['cluster'] == k
    label = etiquetas_cluster.get(k, f'Cluster {k}')
    axes[0].scatter(X_pca[mask.values, 0], X_pca[mask.values, 1],
                    c=COLORES[k % len(COLORES)], label=label,
                    alpha=0.65, s=40, edgecolors='white', linewidth=0.3)

axes[0].set_xlabel(f'PC1 ({var_exp[0]:.1%} varianza)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({var_exp[1]:.1%} varianza)', fontsize=11)
axes[0].set_title(f'Clusters en Espacio PCA 2D\nSilhouette = {sil_final:.3f}', fontweight='bold')
axes[0].legend(title='Perfil de Vulnerabilidad', fontsize=10)

# Heatmap de centroides normalizados
centroides_norm = (centroides - centroides.mean()) / centroides.std()
centroides_norm.index = [etiquetas_cluster.get(i, f'C{i}') for i in centroides_norm.index]
sns.heatmap(centroides_norm.T, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=axes[1])
axes[1].set_title('Perfiles de Clusters\n(valores normalizados: verde=alto, rojo=bajo)', fontweight='bold')
axes[1].set_xlabel('Perfil')

plt.suptitle('Segmentación de Comunas por Vulnerabilidad Digital – K-Means', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, '07_clusters_pca_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# TOP COMUNAS POR CLUSTER
# ============================================================
print('=== PERFILES DE VULNERABILIDAD DIGITAL ===')
for k in sorted(df_model['cluster'].unique()):
    perfil = etiquetas_cluster.get(k, f'Cluster {k}')
    subset = df_model[df_model['cluster'] == k]
    print(f'\n--- {perfil} (n={len(subset)} comunas) ---')
    print(subset[FEATURE_COLS].describe().round(2).loc[['mean','min','max']].to_string())
    print(f'Comunas representativas (mayor población):')
    top = subset.nlargest(5, 'poblacion_2024')[['nombre_comuna', 'poblacion_2024', 'conexiones_x1000hab']]
    print(top.to_string(index=False))

---
# SECCIÓN 7: Modelo de Clasificación – Baseline

## Objetivo
Con las etiquetas de vulnerabilidad generadas por el clustering, entrenamos modelos supervisados para:
1. **Auditar la separabilidad** de los perfiles descubiertos (modelo subrogado).
2. **Identificar** qué variables tienen mayor importancia en la segmentación.

## Esquema de validación (Cero Data Leakage)
Para resolver cualquier riesgo de **fuga de datos (data leakage)**, la partición train/test se realiza **antes** de cualquier normalización, y el escalado `MinMaxScaler` se encapsula dentro de un `Pipeline` de scikit-learn. Así, cada pliegue de validación cruzada ajusta los parámetros de escala exclusivamente con su conjunto de entrenamiento:

```
Dataset completo (346 comunas)
       ↓ split 80/20 estratificado (sin tocar los datos)
  80% TRAIN ─── Pipeline(MinMaxScaler + Classifier) ──► 5-Fold Stratified CV
  20% TEST  ─── Evaluación Final: Accuracy, Precision, Recall, F1
```


In [ ]:
# ============================================================
# PREPARAR DATOS DE CLASIFICACIÓN (CERO DATA LEAKAGE)
# ============================================================
df_cls = df_model[FEATURE_COLS + ['cluster']].dropna()
X_cls = df_cls[FEATURE_COLS].values
y_cls = df_cls['cluster'].values.astype(int)

print(f'Dataset clasificación: {X_cls.shape[0]} comunas | {X_cls.shape[1]} features | {len(set(y_cls))} clases')
print(f'Distribución de clases: {dict(zip(*np.unique(y_cls, return_counts=True)))}')

# Split 80/20 estratificado ANTES de cualquier escalado para evitar Data Leakage
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.20, random_state=RANDOM_SEED, stratify=y_cls
)
print(f'Train: {len(X_train)} comunas | Test: {len(X_test)} comunas')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)


In [ ]:
# ============================================================
# MODELO 1: REGRESIÓN LOGÍSTICA (BASELINE CON PIPELINE)
# ============================================================
print('=== REGRESIÓN LOGÍSTICA (BASELINE) ===')

pipe_lr = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs', random_state=RANDOM_SEED))
])

# Cross-validation en train (el scaler se ajusta estrictamente dentro de cada fold)
cv_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring='f1_weighted')
print(f'CV F1-Score (5-fold sin data leakage): {cv_lr.mean():.4f} ± {cv_lr.std():.4f}')

# Entrenamiento y evaluación en test
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

metricas_lr = {
    'Accuracy':  accuracy_score(y_test, y_pred_lr),
    'Precision': precision_score(y_test, y_pred_lr, average='weighted', zero_division=0),
    'Recall':    recall_score(y_test, y_pred_lr, average='weighted', zero_division=0),
    'F1-Score':  f1_score(y_test, y_pred_lr, average='weighted', zero_division=0),
}

for k, v in metricas_lr.items():
    print(f'  {k}: {v:.4f}')

print('\nReporte detallado:')
print(classification_report(y_test, y_pred_lr, zero_division=0))

joblib.dump(pipe_lr, os.path.join(MODELS, 'logistic_regression.pkl'))
joblib.dump(pipe_lr.named_steps['scaler'], os.path.join(MODELS, 'scaler_clasificacion.pkl'))


In [ ]:
# ============================================================
# MODELO 2: RANDOM FOREST (CON PIPELINE)
# ============================================================
print('=== RANDOM FOREST ===')

pipe_rf = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_split=5,
                                          min_samples_leaf=2, random_state=RANDOM_SEED, n_jobs=-1))
])

cv_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring='f1_weighted')
print(f'CV F1-Score (5-fold sin data leakage): {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')

pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)

metricas_rf = {
    'Accuracy':  accuracy_score(y_test, y_pred_rf),
    'Precision': precision_score(y_test, y_pred_rf, average='weighted', zero_division=0),
    'Recall':    recall_score(y_test, y_pred_rf, average='weighted', zero_division=0),
    'F1-Score':  f1_score(y_test, y_pred_rf, average='weighted', zero_division=0),
}

for k, v in metricas_rf.items():
    print(f'  {k}: {v:.4f}')

print('\nReporte detallado:')
print(classification_report(y_test, y_pred_rf, zero_division=0))

joblib.dump(pipe_rf, os.path.join(MODELS, 'random_forest.pkl'))


In [ ]:
# ============================================================
# VISUALIZACIÓN: MATRICES DE CONFUSIÓN + FEATURE IMPORTANCE
# ============================================================
class_names = [etiquetas_cluster.get(k, f'C{k}') for k in sorted(set(y_cls))]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Matriz LR
for ax, y_pred, titulo in [
    (axes[0], y_pred_lr, 'Regresión Logística (Baseline)'),
    (axes[1], y_pred_rf, 'Random Forest')
]:
    cm_mat = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_mat, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_xlabel('Predicción', fontsize=11)
    ax.set_ylabel('Real', fontsize=11)
    ax.set_title(titulo, fontweight='bold')

# Feature Importance
rf_model = pipe_rf.named_steps['classifier']
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
colors_imp = plt.cm.viridis(np.linspace(0.2, 0.85, len(importances)))
bars = axes[2].barh(importances.index, importances.values, color=colors_imp, edgecolor='white')
for bar, val in zip(bars, importances.values):
    axes[2].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)
axes[2].set_title('Feature Importance – Random Forest\n(¿Qué determina la vulnerabilidad digital?)', fontweight='bold')
axes[2].set_xlabel('Importancia')
axes[2].set_xlim(0, importances.max() * 1.2)

plt.suptitle('Evaluación de Modelos de Clasificación – Vulnerabilidad Digital por Comuna', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, '08_modelos_confusion_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# COMPARATIVA FINAL DE MODELOS
# ============================================================
df_comp = pd.DataFrame({'Reg. Logística': metricas_lr, 'Random Forest': metricas_rf}).T

print('=== COMPARATIVA DE MODELOS ===')
display(df_comp.style.highlight_max(axis=0, color='lightgreen').format('{:.4f}'))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_comp.columns))
w = 0.35
b1 = ax.bar(x - w/2, df_comp.loc['Reg. Logística'], w,
            label='Reg. Logística (Baseline)', color='#3498db', alpha=0.85, edgecolor='white')
b2 = ax.bar(x + w/2, df_comp.loc['Random Forest'], w,
            label='Random Forest', color='#27ae60', alpha=0.85, edgecolor='white')
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_comp.columns, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Comparativa de Modelos – Clasificación de Vulnerabilidad Digital', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, '09_comparativa_modelos.png'), dpi=150, bbox_inches='tight')
plt.show()

mejor = 'Random Forest' if metricas_rf['F1-Score'] > metricas_lr['F1-Score'] else 'Reg. Logística'
print(f'Mejor modelo: {mejor} (F1={max(metricas_rf["F1-Score"], metricas_lr["F1-Score"]):.4f})')

---
# SECCIÓN 8: Resumen, Conclusiones y Próximos Pasos

## Hallazgos del EDA

1. **Desigualdad extrema:** La conectividad varía más de 100x entre comunas → las regiones metropolitanas tienen acceso radicalmente distinto a las zonas rurales extremas.
2. **H2 validada:** Existe correlación negativa entre tasa de pobreza y conectividad → comunas más pobres tienen menos acceso a internet.
3. **Ruralidad como factor clave:** Las comunas clasificadas como rurales (proxy <15.000 hab.) tienen medianas de conectividad significativamente menores (Mann-Whitney p<0.05).
4. **Regiones extremas:** Aysén, Los Lagos y La Araucanía muestran los menores índices de conectividad.

## Resultados del Clustering

| Perfil | Descripción | N° Comunas aproximado |
|--------|-------------|----------------------|
| **Alta Vulnerabilidad** | Alta pobreza, baja conectividad, alta ruralidad | *(ver resultados)* |
| **Vulnerabilidad Media** | Valores intermedios en todas las variables | *(ver resultados)* |
| **Baja Vulnerabilidad** | Alta conectividad, menor pobreza, urbanas | *(ver resultados)* |

## Métricas del Modelo Baseline

| Modelo | CV F1 (5-fold) | Test Accuracy | Test F1 |
|--------|----------------|--------------|--------|
| Reg. Logística | *(ver resultados)* | *(ver resultados)* | *(ver resultados)* |
| Random Forest | *(ver resultados)* | *(ver resultados)* | *(ver resultados)* |

## Próximos pasos (Etapa III)
1. **XGBoost y SVM:** Comparar con más modelos
2. **Optimización de hiperparámetros:** GridSearchCV / RandomizedSearchCV
3. **Interpretabilidad avanzada:** SHAP values para explicaciones individuales por comuna
4. **Datos geoespaciales:** Incorporar shapefiles para mapas interactivos (Folium)
5. **Variable objetivo más refinada:** Construir un índice compuesto de conectividad
6. **Validación más robusta:** RepeatedStratifiedKFold, curvas ROC por clase

In [ ]:
# ============================================================
# RESUMEN EJECUTIVO FINAL
# ============================================================
print('=' * 65)
print('  RESUMEN EJECUTIVO – ETAPA II')
print('  Análisis de la Brecha Digital en Chile')
print('=' * 65)
print()
print('DATOS UTILIZADOS')
print(f'  Fuentes: SUBTEL (conexiones 2015–2026) + INE (pob. 2024) + CASEN (pobreza 2022)')
print(f'  Dataset integrado: {len(df)} comunas x {df.shape[1]} variables')
print(f'  Comunas con datos SUBTEL: {df["conexiones_recientes"].notna().sum()}')
print(f'  Comunas con datos CASEN:  {df["tasa_pobreza_ingresos"].notna().sum()}')
print()
print('CLUSTERING (K-Means)')
print(f'  k óptimo: {N_CLUSTERS} (Silhouette Score: {sil_final:.4f})')
for k, lbl in etiquetas_cluster.items():
    n_com = (df_model['cluster'] == k).sum()
    print(f'  Cluster {k} [{lbl}]: {n_com} comunas')
print()
print('CLASIFICACIÓN (Baseline)')
for modelo, m in [('Reg. Logística', metricas_lr), ('Random Forest', metricas_rf)]:
    print(f'  {modelo}: Accuracy={m["Accuracy"]:.3f} | F1={m["F1-Score"]:.3f}')
print()
print('FIGURAS GENERADAS')
figs = sorted([f for f in os.listdir(FIGURES) if f.endswith('.png')])
for fig_name in figs:
    print(f'  outputs/figures/{fig_name}')
print()
print('MODELOS GUARDADOS')
mods = sorted([f for f in os.listdir(MODELS) if f.endswith('.pkl')])
for m_name in mods:
    print(f'  outputs/models/{m_name}')
print()
print('=' * 65)